# 🎬 Wan 2.2 A14B (כימות 4-bit) — Image to Video · לאה · v2
תוקן 16/10: המודל האמיתי הוא A14B (MoE, 28.6GB ב-fp16) — בגרסה זו הוא נטען בכימות 4-bit (~8GB) כדי להיכנס ל-T4.
**הגדרות:** זמן ריצה ← שנה ← T4 GPU. ואז: ריצה ← הפעל הכל.
⏱ ריצה ראשונה: הורדת המודל ~10 דק' + רינדור 20–50 דק'. אם נתקע בזיכרון — הורד FRAMES ל-49 ו-RES ל-480x480 (מוסבר בתא הרינדור).

In [ ]:
!pip -q install diffusers transformers accelerate safetensors sentencepiece einops bitsandbytes
import torch
assert torch.cuda.is_available(), 'אין GPU! הפעלו: זמן ריצה ← שנה ← T4'
print('✔ GPU:', torch.cuda.get_device_name(0))

In [ ]:
IMAGE_URL = 'https://aidoor.shimi.land/gallery/lea-bikini/lea_bikini_2_fullcanon_seed161007.png'
PROMPT = '''A beautiful young woman with long dark hair dancing gracefully in a modern loft
at golden hour. Natural fluid body movement, hips swaying, hands flowing through the air,
soft cinematic lighting, warm tones, photorealistic, smooth motion.'''
NEGATIVE = 'blurry, low quality, distorted face, deformed hands, extra limbs, watermark, text, jittery flickering'
SEED = 161007
FRAMES = 81; FPS = 16; HEIGHT, WIDTH = 480, 832
STEPS = 20   # אפשר 12 למהירות

## טעינת המודל בכימות 4-bit (~10 דקות, פעם אחת)
ה-transformer (14B MoE) והטקסט-אנקודר (UMT5-XXL) שניהם ב-4-bit — כך נכנסים ל-12.7GB הזיכרון של Colab.

In [ ]:
import torch
from diffusers import WanImageToVideoPipeline, WanTransformer3DModel, AutoencoderKLWan
from diffusers import UniPCMultistepScheduler, BitsAndBytesConfig as BnB4Diff
from transformers import UMT5EncoderModel, T5TokenizerFast, BitsAndBytesConfig as BnB4HF
from diffusers.utils import export_to_video, load_image

MODEL = 'Wan-AI/Wan2.2-I2V-A14B-Diffusers'
q4 = BnB4Diff(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
q4hf = BnB4HF(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

transformer = WanTransformer3DModel.from_pretrained(MODEL, subfolder='transformer', quantization_config=q4)
text_encoder = UMT5EncoderModel.from_pretrained(MODEL, subfolder='text_encoder', quantization_config=q4hf)
tokenizer = T5TokenizerFast.from_pretrained(MODEL, subfolder='tokenizer')
vae = AutoencoderKLWan.from_pretrained(MODEL, subfolder='vae', torch_dtype=torch.float16)
scheduler = UniPCMultistepScheduler.from_pretrained(MODEL, subfolder='scheduler')

pipe = WanImageToVideoPipeline(tokenizer=tokenizer, text_encoder=text_encoder, vae=vae,
                               scheduler=scheduler, image_processor=None, image_encoder=None,
                               transformer=transformer)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
print('✔ המנוע נטען ב-4-bit')

## רינדור — אפשר להתנתק, Colab ממשיך (אל תסגרו את הכרטיסייה דרך 'נהל סשן')

In [ ]:
try:
    img = load_image(IMAGE_URL)
    g = torch.Generator('cuda').manual_seed(SEED)
    out = pipe(image=img, prompt=PROMPT, negative_prompt=NEGATIVE,
               height=HEIGHT, width=WIDTH, num_frames=FRAMES,
               num_inference_steps=STEPS, guidance_scale=5.0, generator=g).frames[0]
    export_to_video(out, 'wan_output.mp4', fps=FPS)
    print('✔ נשמר: wan_output.mp4')
except torch.cuda.OutOfMemoryError:
    print('אזל הזיכרון. עצירת זמן ריצה ← הפעלה מחדש ← הורידו: FRAMES=49, STEPS=12, HEIGHT,WIDTH=480,480')
    raise

In [ ]:
from IPython.display import HTML
HTML('<video width=640 controls loop src="wan_output.mp4"></video>')

In [ ]:
import requests
r = requests.post('https://catbox.moe/user/api.php',
    data={'reqtype':'fileupload'}, files={'fileToUpload': open('wan_output.mp4','rb')})
print('🔗 שלחו את הקישור לשימי פיתוח לאכלוס הגלריה:', r.text if r.ok else 'נכשל: '+r.text)